# Tool + 기본 Agent

In [1]:
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
MODEL_NAME="gemini-3.6-flash"

### Tool 개념

- Tool은 LLM이 외부 세계와 상호작용할 수 있게 해주는 함수이다
- LLM 자체는 텍스트만 생성할 수 있지만, Tool을 통해 웹 검색, 계산, 외부 API 조회 등을 수행할 수 있다
- LLM이 "이 Tool을 호출해야겠다"고 판단하면, Tool 이름과 인자를 반환한다

```
사용자 질문 → LLM이 판단 → Tool 호출 필요?
  → Yes: Tool 이름 + 인자 반환 → Tool 실행 → 결과를 LLM에 전달 → 최종 응답
  → No: 바로 텍스트 응답
```

예를 들어 사용자가 "서울 날씨 알려줘"라고 하면, LLM은 스스로 날씨를 알 수 없다. 대신 "날씨 검색 Tool을 '서울'이라는 인자로 호출해야겠다"고 판단한다. 개발자가 실제로 Tool을 실행하고 결과를 LLM에 돌려주면, LLM이 그 결과를 자연어로 정리하여 답변한다.

---


### @tool 데코레이터

- Python 함수를 LangChain Tool로 변환하는 가장 간단한 방법
- 함수의 docstring이 Tool의 설명(description)이 된다
- `@tool` 데코레이터를 붙이면 일반 함수가 LangChain Tool 객체로 변환된다. LLM은 이 Tool의 `name`, `description`, `args_schema`를 보고 어떤 Tool을 어떤 인자로 호출할지 판단한다. 따라서 **docstring(설명)과 타입 힌트가 매우 중요하다**.

`@tool(parse_docstring=True)`를 사용하고 docstring을 Google 스타일의 `Args:` 형식으로 작성하면, LangChain이 각 parameter의 설명을 자동으로 추출해 `args_schema`의 `description`에 넣어준다. 별도의 Pydantic 스키마 없이도 인자의 의미를 LLM에게 전달할 수 있다.

```python
@tool(parse_docstring=True)
def search_weather(city: str) -> str:
    """도시의 현재 날씨를 검색한다.

    Args:
        city: 날씨를 검색할 도시 이름
    """
```

`search_weather.args_schema.model_json_schema()`로 자동 생성된 parameter description을 확인할 수 있다.


#### Tool 설계 원칙

| 원칙 | 설명 |
|------|------|
| 명확한 이름 | `search_weather` > `func1` |
| 구체적인 설명 | "주어진 도시의 현재 날씨 정보를 검색한다" > "데이터를 가져온다" |
| 타입 힌트 필수 | `city: str` — LLM이 어떤 값을 넣어야 하는지 알 수 있다 |
| 예시 포함 | docstring에 입력 예시를 넣으면 정확도가 높아진다 |
| 에러 메시지 | Tool 실행 실패 시 LLM이 이해할 수 있는 메시지를 반환한다 |

**Tool을 나누는 기준**: API 엔드포인트가 아니라 **LLM이 docstring만 보고 언제 쓸지 판단할 수 있는 단위**로 나눈다. 용도가 다르면 분리하고(검색 vs 상세 조회), 파라미터 하나 차이면 합쳐도 된다. 너무 많으면 선택 정확도가 떨어지고, 너무 합치면 파라미터가 복잡해져서 LLM이 헷갈린다.

In [2]:
@tool(parse_docstring=True)
def search_weather(city: str) -> str:
    """주어진 도시의 현재 날씨를 검색한다.

    Args:
        city: 날씨를 검색할 도시 이름. 예: '서울', '부산'
    """
    weather_data = {
        "서울": "맑음, 22도, 습도 45%",
        "부산": "흐림, 19도, 습도 72%",
        "제주": "비, 17도, 습도 88%",
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

# Tool 정보 확인
print(f"이름: {search_weather.name}")
print(f"설명: {search_weather.description}")
print(f"스키마: {search_weather.args_schema.model_json_schema()}")

# Tool 직접 호출
print(f"\n서울 날씨: {search_weather.invoke({'city': '서울'})}")

이름: search_weather
설명: 주어진 도시의 현재 날씨를 검색한다.
스키마: {'description': '주어진 도시의 현재 날씨를 검색한다.', 'properties': {'city': {'description': "날씨를 검색할 도시 이름. 예: '서울', '부산'", 'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'search_weather', 'type': 'object'}

서울 날씨: 맑음, 22도, 습도 45%


---


### Tool 바인딩

- LLM에 사용할 수 있는 Tool 목록을 알려주는 것
- `bind_tools()`를 사용한다

`bind_tools()`로 Tool을 바인딩하면, LLM은 사용자의 질문을 보고 두 가지 중 하나를 선택한다.

1. **Tool 호출이 필요한 경우** — `tool_calls`에 호출할 Tool 정보를 담아서 반환 (content는 비어 있을 수 있음)
2. **Tool 호출이 불필요한 경우** — 일반 텍스트 응답을 content에 담아서 반환

중요한 점은, `bind_tools()`만으로는 **Tool이 실제로 실행되지 않는다**. LLM은 "이 Tool을 이 인자로 호출해줘"라고 요청할 뿐이고, 실제 실행은 개발자가 해야 한다.

In [3]:
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
llm_with_tools = llm.bind_tools([search_weather])

# Tool 호출이 필요한 경우
response = llm_with_tools.invoke("서울 날씨 알려줘")
print("content:", response.content)        # 비어 있을 수 있음
print("tool_calls:", response.tool_calls)   # Tool 호출 정보

print()

# Tool 호출이 불필요한 경우
response2 = llm_with_tools.invoke("안녕하세요")
print("content:", response2.content)        # 일반 응답
print("tool_calls:", response2.tool_calls)  # 빈 리스트

content: []
tool_calls: [{'name': 'search_weather', 'args': {'city': '서울'}, 'id': 'call_714472', 'type': 'tool_call'}]

content: [{'type': 'text', 'text': '안녕하세요! 무엇을 도와드릴까요?', 'extras': {'signature': 'EsQCCsECARFNMg+uwVNwstZPV8q0a5SJIqjK8QCG5YTKATxitbvmf2ET1TlTsOX+QFZ3syl7S/2pRAWcTDC+wBJ3k2H6LJwnMS6jRYaOGCoXFS9n8CfF1STS8J4r4jsCIxze7FHll6+v+e7fPWhVPoDB4Hqk5s4Juw75VvOOYea6JsGz9NzCAjzbxyWXvsvacc2jcBk2nkjPXnDIraIpr9hsJMifAT7pxmANFVIFF3+o/73ZA0Wvh0+MPCKrs18FlNveHdVPjNlF8HrsYGeYowt8oR8Bwo48wzScoFuQd8bRxU/a/1GVyzgNLH54tKrSE2ouY0vShMiF0A4Zxoo/7NHoKBqk5dS8x1zndoy4TpgzoOs5qxXHC48FDm98k+CKnXs/dUDGCykJ9RoamR7csYdEjuibqwLX7pRfnJw4m+zBYOrnWROt'}}]
tool_calls: []


---


### Function Calling 비교: Google Gen AI SDK vs LangChain

| 비교 항목 | Google Gen AI SDK | LangChain @tool |
|-----------|-------------|------------------|
| Tool 정의 | JSON 스키마 직접 작성 | Python 함수 + docstring |
| 파라미터 | 수동으로 properties 정의 | 타입 힌트에서 자동 추출 |
| 결과 확인 | provider SDK 고유 응답 구조 | `AIMessage.tool_calls`로 표준화 |
| 모델 교체 | API별 코드 재작성 | `ChatGoogleGenerativeAI` → 다른 ChatModel 한 줄 변경 |

---


### Tool 호출 루프

Agent는 모델이 Tool을 요청하는 동안 **모델 호출 → Tool 실행 → `ToolMessage` 추가 → 모델 재호출** 과정을 반복한다. 모델은 Tool을 직접 실행하지 않으므로 애플리케이션이 `tool_calls`를 읽고 해당 Tool을 실행해야 한다.

```
사용자 입력 → 모델 판단 → Tool 실행 → 결과 전달 → 모델 재호출 → 최종 응답
```

In [4]:
tools = [search_weather]
tool_map = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

MAX_ITERATIONS = 5


def run_agent(user_input: str) -> str:
    messages = [HumanMessage(content=user_input)]

    for _ in range(MAX_ITERATIONS):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:
            tool_result = tool_map[tool_call["name"]].invoke(tool_call["args"])
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"],
                )
            )

    return "최대 반복 횟수를 초과했습니다."

In [5]:
# 다양한 질문으로 테스트
questions = [
    "서울 날씨 어때?",
    "서울이랑 부산 날씨 비교해줘",
    "안녕하세요!",
]

for q in questions:
    print(f"\n사용자: {q}")
    answer = run_agent(q)
    print(f"Agent: {answer}")


사용자: 서울 날씨 어때?
Agent: [{'type': 'text', 'text': '현재 서울의 날씨는 **맑음**이며, 기온은 **22℃**, 습도는 **45%**입니다.', 'extras': {'signature': 'Eo4CCosCARFNMg8JE0+INWux/evrOh8MQqjfV2eoeme+dv5Bjhr0D5y5gveZa/O3Azu3sws7o3tPr9vcSYLLk41qoxZZH/xIVmU6GbqdWmaEaTemm7QkHI2ZjxhO3hznJU8HsTCHKgoqMKEfEZlYEayK8TRFhQ9lDWGRvZtl7TOfGQi74l/GcXkRzVBrNsJmlGOCvR04EthXCp6dJgbFjR0MVi7HiVDBthpdHAFWRp4w+WKbFotrn16nmVIX85Q88Uw38iaEg3YSQX+bSiOCTE10V6+i21dLg7ux7w7q1dAJIuuzLennpXVuG6JoPaxcr0ATDLBT7iFb24P6yJI3fAIF3VN8o6hJuu44Vcqd4vDa'}}]

사용자: 서울이랑 부산 날씨 비교해줘
Agent: [{'type': 'text', 'text': '서울과 부산의 현재 날씨 비교 정보입니다.\n\n* **서울**: 맑음, **22°C**, 습도 45%\n* **부산**: 흐림, **19°C**, 습도 72%\n\n**요약**:\n서울은 부산보다 3°C 높고 맑은 날씨인 반면, 부산은 흐리고 습도가 비교적 높은 편입니다.', 'extras': {'signature': 'EqICCp8CARFNMg/hkJkaXYM6sRaK6sM8D84WdJv3lldBf+M8gfET1F0yhfmBwbc3fUFkMtgVv5UEd6wkkm54G/r8TVgkvEj5PEJlkoXA7OMxBryEyuyfGzBhZogFo5KAvFbOEINeKa/EPsdpp4k7v0Bu0MsvDYmrOFNT5zFiva1O6qvfKd6HcYwnKm5teghHeJsm5YRZDhWob4eMdjQxgDhOaZKVxt6+5vw0ksujtAp5Qxh5LTpTQfI21qCTbEVE3aiU1h7K9kH

> **참고**: 모델은 한 번에 여러 Tool을 요청하거나 Tool 결과를 보고 추가 Tool을 요청할 수 있다. 반복 횟수를 제한해 무한 호출을 방지해야 한다. 이 반복 구조는 이후 LangGraph에서 `ToolNode`와 `tools_condition`으로 구성한다.

---

### 실습 문제

#### 영화 목록과 상세 조회 Agent

TMDB API와 LangChain Tool을 사용해 영화 목록을 조회하고, 후속 질문에서 특정 영화의 상세 정보를 조회하는 Agent를 구현하세요.

```.env
TMDB_API_KEY=...
```

다음 Tool을 구현합니다.

| Tool | 역할 |
|---|---|
| `get_movie_list` | 인기, 현재 상영, 개봉 예정 목록에서 영화 ID, 제목, 개봉일을 조회한다 |
| `get_movie_detail` | 영화 ID로 줄거리, 장르, 러닝타임 등 상세 정보를 조회한다 |

목록 Tool은 상세 정보를 포함하지 않습니다. 후속 질문에 답하려면 목록 결과의 영화 ID를 사용해 `get_movie_detail`을 호출해야 합니다.

다음 두 질문을 **같은 세션**에서 순서대로 실행합니다.

```text
현재 상영 중인 영화 5개를 알려줘.
그중 첫 번째 영화의 줄거리와 러닝타임을 알려줘.
```

다른 세션에서 바로 `"그중 첫 번째 영화의 줄거리를 알려줘."`라고 질문했을 때 이전 목록을 알 수 없다고 답하는지도 확인하세요.